# OpenVals — AI Model Validation from Zero
## Episode 02: How Do We Know an AI Model Actually Works?

A model score is not automatically evidence of trustworthiness.

This notebook treats classification performance as a scientific measurement problem. It combines exact definitions, rendered equations, direct calculations, controlled experiments, uncertainty analysis, and simulation.

### Central principle

> **A metric is a number. Evidence requires context.**


## 1. Binary Classification

For a binary classifier, the true class and predicted class each take one of two possible values.

The four possible outcomes are:

- **True Positive (TP):** actual positive, predicted positive
- **False Negative (FN):** actual positive, predicted negative
- **False Positive (FP):** actual negative, predicted positive
- **True Negative (TN):** actual negative, predicted negative

The total number of evaluated observations is:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

print("NumPy:", np.__version__)
print("Environment ready.")


## 2. The 95% Accuracy Example

Consider 1,000 observations:

- 950 negative cases
- 50 positive cases

Now consider a classifier that predicts **negative for every single observation**.

Its confusion-matrix counts are:


Accuracy is defined as:


Substituting the values:


In [ ]:
TP, FN, FP, TN = 0, 50, 0, 950
N = TP + TN + FP + FN
accuracy = (TP + TN) / N

print("N        =", N)
print("TP       =", TP)
print("FN       =", FN)
print("FP       =", FP)
print("TN       =", TN)
print("Accuracy =", f"{accuracy:.2%}")


## 3. Recall Reveals a Different Property

Recall measures the fraction of actual positive cases that were correctly identified.


For this classifier:


The same classifier therefore has:

- **Accuracy = 95%**
- **Recall = 0%**

There is no mathematical contradiction. The two metrics quantify different properties of the same classifier.


In [ ]:
recall = TP / (TP + FN)
print("Recall =", f"{recall:.2%}")


## 4. Confusion Matrix

The confusion matrix records the four binary classification outcomes.


In [ ]:
cm = np.array([[TP, FN],
               [FP, TN]])

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)
ax.set_xticks([0, 1], labels=["Predicted Positive", "Predicted Negative"])
ax.set_yticks([0, 1], labels=["Actual Positive", "Actual Negative"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)

ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 5. Core Classification Metrics


### Accuracy


### Precision


Precision is mathematically undefined when the model predicts no positive cases, because then the denominator is zero.


### Recall / Sensitivity / True Positive Rate


### Specificity / True Negative Rate


### False Positive Rate


### False Negative Rate


### F1 Score


In [ ]:
def safe_divide(a, b):
    return np.nan if b == 0 else a / b

def classification_metrics(tp, tn, fp, fn):
    accuracy = safe_divide(tp + tn, tp + tn + fp + fn)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    fpr = safe_divide(fp, fp + tn)
    fnr = safe_divide(fn, fn + tp)

    if np.isnan(precision) or np.isnan(recall) or precision + recall == 0:
        f1 = np.nan
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "FPR": fpr,
        "FNR": fnr,
        "F1": f1,
    }

metrics = classification_metrics(TP, TN, FP, FN)

for name, value in metrics.items():
    text = "undefined" if np.isnan(value) else f"{value:.2%}"
    print(f"{name:12s} = {text}")


## 6. Class Imbalance

Let the negative-class prevalence be:


and the positive-class prevalence be:


A classifier that always predicts the majority class achieves:


This means raw accuracy can become numerically impressive even when the classifier has no useful ability to identify the minority class.


In [ ]:
distributions = [(950,50),(800,200),(500,500),(990,10),(999,1)]

print("NEGATIVE  POSITIVE   ACCURACY   RECALL")
print("-" * 43)

rows = []
for negatives, positives in distributions:
    tp, fn, fp, tn = 0, positives, 0, negatives
    total = negatives + positives
    acc = (tp + tn) / total
    rec = tp / (tp + fn)
    rows.append((negatives, positives, acc, rec))
    print(f"{negatives:8d}  {positives:8d}   {acc:8.2%}   {rec:6.2%}")


In [ ]:
labels = [f"{n}/{p}" for n,p,_,_ in rows]
values = [a for _,_,a,_ in rows]

plt.figure(figsize=(8,5))
plt.bar(labels, values)
plt.ylim(0, 1.05)
plt.xlabel("Negative / Positive observations")
plt.ylabel("Accuracy")
plt.title("Accuracy of an Always-Negative Classifier")
plt.grid(axis="y", alpha=0.25)
plt.show()


## 7. Same Accuracy, Different Behaviour

Consider two models evaluated on the same balanced dataset of 1,000 observations.

Model A:


Model B:


Both models have:


Their operational behaviour is nevertheless different because their error distributions differ.


In [ ]:
model_A = classification_metrics(tp=450, tn=450, fp=50, fn=50)
model_B = classification_metrics(tp=400, tn=500, fp=0, fn=100)

print("METRIC         MODEL A     MODEL B")
print("-" * 36)

for metric in model_A:
    a, b = model_A[metric], model_B[metric]
    a = "undefined" if np.isnan(a) else f"{a:.2%}"
    b = "undefined" if np.isnan(b) else f"{b:.2%}"
    print(f"{metric:12s}   {a:>8s}   {b:>8s}")


## 8. Balanced Accuracy

Balanced accuracy gives equal weight to sensitivity and specificity.


Using the underlying confusion-matrix terms:


For the always-negative classifier:


Therefore:


In [ ]:
specificity = TN / (TN + FP)
balanced_accuracy = (recall + specificity) / 2

print("Raw accuracy      =", f"{accuracy:.2%}")
print("Balanced accuracy =", f"{balanced_accuracy:.2%}")


## 9. Sample Size and Sampling Uncertainty

Assume each evaluated prediction is an independent Bernoulli trial with probability \(p\) of being correct.

Observed accuracy is the sample proportion:


Under the Bernoulli model:


Sampling uncertainty therefore decreases proportionally to the inverse square root of sample size.


## 10. Same Point Estimate, Different Uncertainty

The following two evaluations both report 90% accuracy:


The plug-in estimate of standard error is:


In [ ]:
def se_proportion(p_hat, n):
    return sqrt(p_hat * (1 - p_hat) / n)

examples = [(9,10),(90,100),(900,1000),(900000,1000000)]

print("CORRECT / N       p-hat        Estimated SE")
print("-" * 46)
for correct, n in examples:
    p_hat = correct / n
    se = se_proportion(p_hat, n)
    print(f"{correct:>7}/{n:<10} {p_hat:8.4f}      {se:.6f}")


## 11. Wilson Score Confidence Interval

A confidence interval quantifies uncertainty in estimating a population proportion from a finite sample.

For a confidence level \(1-lpha\), let the corresponding standard-normal critical value be \(z\).

The Wilson interval center is:


The Wilson half-width is:


The confidence interval is:


In [ ]:
def wilson_interval(successes, n, z=1.959963984540054):
    p_hat = successes / n
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denominator
    half_width = (
        z / denominator
        * sqrt(
            p_hat * (1-p_hat) / n
            + z**2 / (4*n**2)
        )
    )
    return center - half_width, center + half_width

print("OBSERVED        ACCURACY       95% WILSON INTERVAL")
print("-" * 57)

for successes, n in examples:
    p_hat = successes / n
    low, high = wilson_interval(successes, n)
    print(
        f"{successes:>7}/{n:<10} "
        f"{p_hat:8.2%}       "
        f"[{low:.2%}, {high:.2%}]"
    )


## 12. Monte Carlo Simulation of Sampling Variability

Assume the true probability of a correct prediction is 0.90.

For a sample of size \(n\), the number of correct predictions follows a binomial distribution:


In [ ]:
rng = np.random.default_rng(42)
true_p = 0.90
repetitions = 10000
sample_sizes = [10, 100, 1000]

simulation_results = {}

for n in sample_sizes:
    correct = rng.binomial(n=n, p=true_p, size=repetitions)
    observed = correct / n
    simulation_results[n] = observed

    print(
        f"n={n:4d} | "
        f"mean={observed.mean():.4f} | "
        f"sample SD={observed.std(ddof=1):.4f}"
    )


In [ ]:
for n in sample_sizes:
    values = simulation_results[n]

    plt.figure(figsize=(8,5))
    plt.hist(values, bins=30, edgecolor="black")
    plt.axvline(true_p, linestyle="--", linewidth=2)
    plt.xlabel("Observed Accuracy")
    plt.ylabel("Frequency")
    plt.title(f"Sampling Distribution of Accuracy, n={n}")
    plt.grid(alpha=0.2)
    plt.show()


Theoretical standard deviation:


In [ ]:
for n in sample_sizes:
    theoretical = sqrt(true_p * (1-true_p) / n)
    empirical = simulation_results[n].std(ddof=1)
    print(
        f"n={n:4d} | "
        f"Theoretical SD={theoretical:.5f} | "
        f"Simulation SD={empirical:.5f}"
    )


## 13. Error Costs

Accuracy does not encode the possibility that false positives and false negatives have different consequences.

A simple cost-sensitive loss is:


When the cost of a false negative is substantially larger than the cost of a false positive, a model with fewer false negatives may be preferable even if its raw accuracy is lower.

The cost values must come from the application context.


In [ ]:
def weighted_error(fp, fn, cost_fp=1.0, cost_fn=1.0):
    return cost_fp * fp + cost_fn * fn

cost_fp = 1
cost_fn = 10

loss_A = weighted_error(fp=50, fn=50, cost_fp=cost_fp, cost_fn=cost_fn)
loss_B = weighted_error(fp=0, fn=100, cost_fp=cost_fp, cost_fn=cost_fn)

print("Cost(FP) =", cost_fp)
print("Cost(FN) =", cost_fn)
print("Model A weighted loss =", loss_A)
print("Model B weighted loss =", loss_B)


## 14. What a Performance Claim Must Specify

A statement such as **“Accuracy = 95%”** is incomplete without the conditions under which it was obtained.

Scientific interpretation can require:

- metric definition
- sample size
- class distribution
- sampling process
- decision threshold
- error structure
- uncertainty
- intended application
- distribution shift


## 15. Statistical Evidence Is Not Mathematical Proof

Mathematical proof and empirical model validation are fundamentally different.

A theorem can be established deductively under axioms and assumptions.

Model evaluation is based on finite observations generated under particular conditions.


The objective of model validation is usually to quantify evidence, uncertainty, limitations, and conditions of use rather than to prove that a model will always work.


## 16. Scientific Validation Chain


A trust decision is not mathematical certainty. It is a decision supported by measurable evidence under stated assumptions and conditions.


## 17. OpenVals Perspective

A production AI system may require evidence across multiple dimensions, including performance, reliability, robustness, safety, consistency, latency, cost, and drift.

The broader validation question is:


This is the scientific direction behind OpenVals.


## 18. Episode 02 Summary

A classifier can report 95% accuracy while achieving 0% recall on the positive class.

Equal accuracy does not imply equal error behaviour.


Under the Bernoulli model:


Sample size therefore affects uncertainty.

Class imbalance can make raw accuracy misleading.

Performance evaluation requires both mathematical measurement and scientifically justified interpretation.

### Central conclusion


### Next Episode

**Episode 03 — Probability, Uncertainty, and Why AI Predictions Are Not Certainties**
